In [ ]:
import common_functions

import numpy as np
from scipy.stats import norm

In [4]:
def calc_dust_counts_fraction(texture_ds, dust_df, texture_dict):

    # Extract the soil texture DataArray
    texture_da = texture_ds.GLDAS_soiltex

    # Flatten the raster to 1D for counting
    texture_flat = texture_da.values.flatten()

    # Count full domain occurrences
    full_counts = {k: np.sum(texture_flat == k) for k in texture_dict.keys()}
    total_full = sum(full_counts.values())
    full_fraction = {k: v / total_full for k, v in full_counts.items()}

    # Subset raster at dust point locations
    # Round coordinates to nearest grid point
    dust_textures = []
    for lon, lat in zip(dust_df["longitude"], dust_df["latitude"]):
        # Select nearest pixel
        val = texture_da.sel(
            lon=lon, lat=lat, method="nearest"
        ).values
        dust_textures.append(val)

    dust_counts = {k: dust_textures.count(k) for k in texture_dict.keys()}
    total_dust = sum(dust_counts.values())
    dust_fraction = {k: v / total_dust for k, v in dust_counts.items()}

    return dust_counts, dust_fraction, full_counts, full_fraction

In [5]:
def proportion_ttest(x1, n1, x2, n2):
    """
    Two-sample test for difference in proportions.
    Returns (t_stat, p_value).
    """

    if n1 == 0 or n2 == 0:
        return np.nan, np.nan

    p1 = x1 / n1
    p2 = x2 / n2

    # pooled proportion
    p_pool = (x1 + x2) / (n1 + n2)

    # standard error
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

    if se == 0:
        return np.nan, np.nan

    t_stat = (p1 - p2) / se

    # two-sided p-value
    p_value = 2 * (1 - norm.cdf(abs(t_stat)))

    return t_stat, p_value

In [6]:
def print_ttest_results(dust_counts, dust_fraction, full_counts, full_fraction, texture_dict):
    print("\n===== Soil Texture Significance Tests =====")
    total_dust = sum(dust_counts.values())
    total_full = sum(full_counts.values())

    for k in texture_dict.keys():
        t_stat, p_val = proportion_ttest(
            dust_counts[k], total_dust,
            full_counts[k], total_full
        )

        significance = ""
        if p_val < 0.001:
            significance = "***"
        elif p_val < 0.01:
            significance = "**"
        elif p_val < 0.05:
            significance = "*"

        print(
            f"{texture_dict[k]:20s} | "
            f"dust={dust_fraction[k]:.3f} "
            f"full={full_fraction[k]:.3f} | "
            f"t={t_stat:7.3f}  p={p_val:.3e} {significance}"
        )

    return

In [ ]:
def open_gldas_file(gldas_path):
    ds = xr.open_dataset(gldas_path)
    return ds

In [ ]:
def filter_to_region(ds, location_name):

    lat_min, lat_max, lon_min, lon_max = common_functions._get_coords_for_region(location_name)

    filtered_ds = ds.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    return filtered_ds

In [ ]:
gldas_path = "data/raw/gldas_soil_texture/GLDASp5_soiltexture_025d.nc4"
location_name = "American Southwest"
texture_ds = open_gldas_file(gldas_path)
texture_ds = filter_to_region(texture_ds, location_name)

dust_df = pd.read_csv("DATA/processed/3_dust_points_vars_2026-04-21.csv")

_, _, texture_dict = common_functions.get_texture_map_features()

dust_counts, dust_fraction, full_counts, full_fraction = calc_dust_counts_fraction(texture_ds, dust_df, texture_dict)
print_ttest_results(dust_counts, dust_fraction, full_counts, full_fraction, texture_dict)